## Recon contract for `outputs/250_recon` (my local folder sturcture)

### 0) Purpose

`251_recon_shared_data` is the **single canonical output location** for *derived* electrode-localization artifacts used by plotting, atlas mapping, and group-average visualizations. It is **regenerable** and should be treated as pipeline output, not raw data.

---
## Part 1: Generate and save the trsRAS contact coordonates in my drive using LEPTOVOX. 

### Per patient structure of the shared folder to collaborators
```

PAT_XXXX/
  BIDS/
    sub-XXXX_electrodes.tsv
    (... possibly other BIDS files later)
  elec_recon/
    electrodeNames.txt (or .csv / .tsv)
    LEPTO.*          # coordinates in lepto space
    LEPTOVOX.*       # coordinates in voxel/MRI space
    (... any additional recon files)
  label/
    lh.aparc.annot
    lh.aparc.DKTatlas.annot
    lh.aparc.a2009s.annot
    rh.aparc.annot
    rh.aparc.DKTatlas.annot
    rh.aparc.a2009s.annot
  mri/
    brainmask.mgz
    aparc+aseg.mgz
    aparc.a2009s+aseg.mgz
    wmparc.mgz
    transforms/
      (registartion matrices; e.g. taliarach.xfm, CT→MRI, etc.)
  surf/ (no PAT_XXX.PIAL files though(
    lh.pial
    rh.pial
    lh.inflated
    rh.inflated
    lh.sphere.reg
    rh.sphere.reg
    lh.curv
    rh.curv
    (... standard FreeSurfer surf files)

fsaverage/ ... (folder on same line as patient folders)
```


## Generate the csv files and the pngs

TO KEEP IN MIND: I had too apply a flip to the k axis [z axis in some standards] because the superior/inferior slice direction was inverted in the LEPTOVOX concention. I looped through multiple perms and flips (take a look at the pngs in the following drive link to compare). https://drive.google.com/drive/folders/1P3B-8-zxNVJe2gxwAYV4mRYBHlq-ed8q?usp=sharing


#### What this code does +/-:
- drop the two header lines in electrodeNames

- read PAT_XXXX.LEPTOVOX

- auto-handle 0-based vs 1-based indexing

- apply k-flip only

- convert voxel→tkrRAS

- compute dist-to-pial + hemisphere

- save CSV + mosaic for all patients

In [ ]:
# ============================================================
# ONE CELL: Fixed LEPTOVOX convention
#   perm  = (0,1,2)  (XYZ as-is)
#   flips = (0,0,1)  (flip k only: k' = (dim_z-1)-k)
#
# Batch over all PAT_* in SHARED_ROOT and write:
#   OUT_ROOT/PAT_XXXX/glassbrain/coords/PAT_XXXX_contacts_tkrRAS.csv
#   OUT_ROOT/PAT_XXXX/glassbrain/png/PAT_XXXX_mosaic_LEPTOVOX.png
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree


# -------------------------
# CONFIG
# -------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")  # change to where your PAT_XXX folders are
OUT_ROOT    = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon") # Change to where you want the data to be saved, in drive format

# Fixed convention you determined:
PERM  = (0, 1, 2)            # use columns as-is
FLIPS = (False, False, True) # flip k only

# render params
VIEWS = ("left", "frontal", "right")
WINDOW_SIZE = (1200, 1000)
TRANSPARENT_BG = True
BRAIN_COLOR = "#ead6db"
BRAIN_OPACITY = 0.25
POINT_COLOR = "purple"
POINT_SIZE = 10
POINT_OPACITY = 0.9

OVERWRITE_CSV = True
OVERWRITE_PNG = True

# Helpers: 
#---------
def read_electrode_lines_drop2(path: Path) -> list[str]:
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 3:
        raise ValueError(f"electrodeNames too short: {path}")
    return lines[2:]  # drop timestamp + header

def parse_name_and_hemi(lines: list[str]):
    """
    lines look like: 'FPG1 D L'
    returns:
      name_raw (full line), name (first token), hemi_expected (L/R/None)
    """
    names_raw, names_clean, hemi = [], [], []
    for ln in lines:
        parts = ln.split()
        nm = parts[0] if len(parts) else ln
        h = None
        if len(parts):
            last = parts[-1].upper()
            if last in ("L", "R"):
                h = last
        names_raw.append(ln)
        names_clean.append(nm)
        hemi.append(h)
    return np.array(names_raw, dtype=object), np.array(names_clean, dtype=object), np.array(hemi, dtype=object)

def read_leptovox_xyz(path: Path) -> np.ndarray:
    rows = []
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        t = ln.strip()
        if not t or t.startswith("#"):
            continue
        parts = t.replace(",", " ").split()
        if len(parts) < 3:
            continue
        try:
            rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
        except ValueError:
            continue
    if not rows:
        raise ValueError(f"No numeric rows found in {path}")
    return np.asarray(rows, dtype=float)

def pick_mgz(subj_dir: Path) -> Path:
    for cand in ["brainmask.mgz", "T1.mgz", "orig.mgz"]:
        p = subj_dir / "mri" / cand
        if p.is_file():
            return p
    raise FileNotFoundError(f"Missing brainmask/T1/orig in {subj_dir/'mri'}")



# Geometry helpers
# ----------------
def voxel_to_tkr(points_ijk: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    n = points_ijk.shape[0]
    ijk_h = np.c_[points_ijk, np.ones(n)]
    tkr_h = (vox2ras_tkr @ ijk_h.T).T
    return tkr_h[:, :3]

def apply_voxel_flips(ijk: np.ndarray, vol_shape, flips=(False, False, True)) -> np.ndarray:
    dims = np.array(vol_shape, dtype=float)
    out = ijk.copy()
    for ax, do_flip in enumerate(flips):
        if do_flip:
            out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
    return out

def nearest_pial_metrics(points_tkr: np.ndarray, lh_v: np.ndarray, rh_v: np.ndarray):
    kdl = cKDTree(lh_v)
    kdr = cKDTree(rh_v)
    dl, _ = kdl.query(points_tkr, k=1, workers=-1)
    dr, _ = kdr.query(points_tkr, k=1, workers=-1)
    is_left = dl <= dr
    dist = np.minimum(dl, dr)
    return dist, is_left



# Rendering helpers
# -------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def set_view(pl, view):
    view = view.lower()
    if view == "left":
        pl.view_yz(negative=True)
    elif view == "frontal":
        pl.view_xz(negative=False)
    elif view == "right":
        pl.view_yz(negative=False)
    else:
        raise ValueError(view)
    pl.camera.zoom(1.15)

def render_view(lh_mesh, rh_mesh, points_tkr, view):
    pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    pl.set_background("white")
    pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
                  point_size=POINT_SIZE, opacity=POINT_OPACITY)
    set_view(pl, view)
    img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
    pl.close()
    return img

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    imgs = [im[:H] for im in imgs]
    return np.concatenate(imgs, axis=1)


# -------------------------
# Subject FreeSurfer dir routing (cohort-aware)
#   PAT_xxxx -> SHARED_ROOT/PAT_xxxx               (read-only from #SHARE)
#   ELxxx    -> BERN_RECON_ROOT/elxxx              (Bern reconstruction tree)
# -------------------------
BERN_RECON_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")

def subj_dir_for(pid: str) -> Path:
    pid = str(pid)
    if pid.startswith("PAT_"):
        return SHARED_ROOT / pid
    if pid.upper().startswith("EL"):
        return BERN_RECON_ROOT / pid.lower()
    raise ValueError(f"Unrecognized patient cohort prefix: {pid}")

def _try_paths(parent: Path, *candidates):
    """Return the first existing file from candidates inside parent, else None."""
    for name in candidates:
        p = parent / name
        if p.is_file():
            return p
    return None

# Per patient
# -------------------------
def export_and_mosaic_patient(pid: str):
    pid = str(pid)
    subj_dir = subj_dir_for(pid)
    elec_dir = subj_dir / "elec_recon"

    # Try uppercase and lowercase pid in filename — Bern's convention varies.
    names_path = _try_paths(elec_dir,
                            f"{pid}.electrodeNames", f"{pid.lower()}.electrodeNames")
    vox_path   = _try_paths(elec_dir,
                            f"{pid}.LEPTOVOX",       f"{pid.lower()}.LEPTOVOX")
    if names_path is None:
        raise FileNotFoundError(f"{pid}: missing electrodeNames in {elec_dir}")
    if vox_path is None:
        raise FileNotFoundError(f"{pid}: missing LEPTOVOX in {elec_dir}")

    mgz = pick_mgz(subj_dir)
    img = nib.load(str(mgz))
    vox2ras_tkr = img.header.get_vox2ras_tkr()
    vol_shape = img.shape[:3]

    # surfaces
    lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_mesh = make_mesh(lh_v, lh_f)
    rh_mesh = make_mesh(rh_v, rh_f)

    # names
    lines = read_electrode_lines_drop2(names_path)
    name_raw, name_clean, hemi_expected = parse_name_and_hemi(lines)

    # leptovox coords
    pts = read_leptovox_xyz(vox_path)
    if pts.shape[0] != len(name_raw):
        raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(name_raw)})")

    # 0/1-based detection
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    dims = np.array(vol_shape, dtype=float)

    looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
    looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()

    pts_ijk = pts.copy()
    index_mode = "0-based (assumed)"
    if looks_one_based and not looks_zero_based:
        pts_ijk -= 1.0
        index_mode = "1-based->0-based"

    # apply fixed perm + flips
    pts_ijk = pts_ijk[:, PERM]  # (0,1,2) = no-op, kept for explicitness
    pts_ijk = apply_voxel_flips(pts_ijk, vol_shape, FLIPS)  # flip k only

    # voxel -> tkrRAS
    pts_tkr = voxel_to_tkr(pts_ijk, vox2ras_tkr)

    # dist + hemi prediction
    dist_mm, pred_is_left = nearest_pial_metrics(pts_tkr, lh_v, rh_v)

    # outputs
    out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
    out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
    out_coords.mkdir(parents=True, exist_ok=True)
    out_pngdir.mkdir(parents=True, exist_ok=True)

    out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
    out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

    # CSV
    if OVERWRITE_CSV or (not out_csv.is_file()):
        df_out = pd.DataFrame({
            "name_raw": name_raw,
            "name": name_clean,
            "hemi_expected": hemi_expected,
            "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
            "pred_isLeft": pred_is_left.astype(int),
            "dist_to_pial_mm": np.round(dist_mm, 2),
            "source_space": "tkrRAS",
            "source_provenance": f"LEPTOVOX; {index_mode}; perm={PERM}; flips={tuple(int(b) for b in FLIPS)} (flip k only)",
            "leptovox_file": str(vox_path),
            "mgz_used": str(mgz),
        })
        df_out.to_csv(out_csv, index=False)

    # Mosaic
    if OVERWRITE_PNG or (not out_png.is_file()):
        imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
        mosaic = stitch_horiz(imgs)
        iio.imwrite(out_png, mosaic)

    # quick stats
    med_dist = float(np.median(dist_mm))
    pct_left = float(np.mean(pred_is_left) * 100.0)

    return {
        "pid": pid,
        "status": "OK",
        "n_contacts": int(len(name_raw)),
        "median_dist_to_pial_mm": med_dist,
        "pct_pred_left": pct_left,
        "index_mode": index_mode,
        "csv": str(out_csv),
        "png": str(out_png),
    }


# -------------------------
# Batch run all PAT_* (Careful, this runs ALL patients in the shared folder. make another patient_ids variable with list of patient names ["PAT_3066","" ...]
# -------------------------
# PAT_xxxx from the shared LEPTOVOX dir
PAT_PATIENTS = sorted([p.name for p in SHARED_ROOT.iterdir()
                       if p.is_dir() and p.name.startswith("PAT_")])

# ELxxx — hardcoded list to control which ones run (Bern recon dirs use
# lowercase 'elXXX' but pid stays uppercase 'ELXXX' for output naming).
EL_PATIENTS = [
    "EL030","EL033","EL034","EL035","EL036","EL037","EL038",
    "EL039","EL040","EL042","EL043","EL044","EL045",
]

patient_ids = PAT_PATIENTS + EL_PATIENTS

rows = []
for pid in patient_ids:
    try:
        rows.append(export_and_mosaic_patient(pid))
    except Exception as e:
        rows.append({"pid": pid, "status": "ERROR", "error": str(e)})

df_status = pd.DataFrame(rows)
df_status


## 250_recon/fsaverage folder outputs: Across all patients

In [ ]:
# ============================================================
# Surface-based group mapping: patient → fsaverage
#
# Three things this cell does (per patient):
#   1) Project surface contacts (is_wm == 0):
#        - PAT (HUG, LEPTOVOX): subject pial → spherical registration → fsaverage
#        - EL  (BERN, Lookup):   subject tkrRAS → talairach affine → fsaverage tkrRAS
#   2) Project depth contacts (is_wm == 1):
#        - Always talairach affine (no snap to cortex). Keeps contact at its
#          fsaverage-MNI volume position rather than collapsing to nearest pial
#          vertex — the previous behaviour mis-placed every hippocampal /
#          amygdalar / deep-white-matter contact onto the surface.
#   3) Look up Yeo 2011 functional networks (7 and 17) at the nearest
#      fsaverage pial vertex for cortical contacts. Depth contacts get
#      yeo7_network = yeo17_network = "WhiteMatter".
#
# Outputs (under OUT_ROOT/fsaverage/coords/):
#   {pid}_contacts_fsaverage.csv               (per patient)
#   ALL_PATIENTS_contacts_fsaverage.csv         (with WM contacts)
#   ALL_PATIENTS_contacts_fsaverage_nowm.csv    (is_wm == 0 only — what 252
#                                                reads when KEEP_WM=False)
#
# Schema:
#   patient, cohort, name, name_raw, hemi, x, y, z, is_wm,
#   is_cortical, dist_to_pial_mm,
#   yeo7_network, yeo17_network
# ============================================================

import re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry, read_annot
from scipy.spatial import cKDTree
import pyvista as pv

# Make `functions/` importable for lf_recon_shared (the EL pathway)
sys.path.insert(0, str(Path('.').resolve()))
sys.path.insert(0, str((Path('.') / 'functions').resolve()))
from functions import lf_recon_shared as rs

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
BERN_RECON_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")
OUT_ROOT    = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon")
FSAVERAGE   = SHARED_ROOT / "fsaverage"

def subj_dir_for(pid: str) -> Path:
    """Subject FreeSurfer dir router. PATs sit under SHARED_ROOT; ELs under
    the Bern Reconstruction tree (lowercase folder names there)."""
    pid = str(pid)
    if pid.startswith("PAT_"):
        return SHARED_ROOT / pid
    if pid.upper().startswith("EL"):
        return BERN_RECON_ROOT / pid.lower()
    raise ValueError(f"Unrecognized cohort: {pid}")

# ------------------------------------------------------------
# LOAD FSAVERAGE SURFACES (used by PAT spherical pathway AND Yeo lookup)
# ------------------------------------------------------------
fs_lh_v, fs_lh_f = read_geometry(str(FSAVERAGE / "surf" / "lh.pial"))
fs_rh_v, fs_rh_f = read_geometry(str(FSAVERAGE / "surf" / "rh.pial"))
fs_lh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "lh.sphere.reg"))
fs_rh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "rh.sphere.reg"))
fs_lh_sphkd  = cKDTree(fs_lh_sph)    # spherical (PAT projection)
fs_rh_sphkd  = cKDTree(fs_rh_sph)
fs_lh_pialkd = cKDTree(fs_lh_v)      # pial (Yeo nearest-vertex + dist-to-pial)
fs_rh_pialkd = cKDTree(fs_rh_v)

# ------------------------------------------------------------
# LOAD YEO 2011 ANNOTS (7 + 17 networks) from fsaverage/label/
# ------------------------------------------------------------
def _decode_annot_names(names):
    return [n.decode("utf-8") if isinstance(n, (bytes, bytearray)) else str(n) for n in names]

# Where to look for Yeo 2011 annot files (FreeSurfer ships these by
# default at $FREESURFER_HOME/subjects/fsaverage/label). Search order
# (first hit wins):
#   1. $FREESURFER_HOME/subjects/fsaverage/label/  (canonical FS install)
#   2. $SUBJECTS_DIR/fsaverage/label/              (alternate FS install)
#   3. BERN_RECON_ROOT/fsaverage/label/            (Bern reconstruction
#      tree — the primary internal source on this server)
#   4. OUT_ROOT/fsaverage/label/                   (analysis repo — commit
#      them here if you want them tracked alongside the recon outputs)
#   5. SHARED_ROOT/fsaverage/label/                (#SHARE folder — READ
#      ONLY fallback. Files have been there since 2011; we never write
#      into this directory.)
#
# Explicitly NOT searched: MNE's local fsaverage cache
# (~/mne_data/MNE-fsaverage-data/) — per user policy.
BERN_RECON_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")

YEO_ANNOT_SEARCH = []
_fs_home = os.environ.get("FREESURFER_HOME")
if _fs_home and _fs_home not in (".", "/", "\\", "S:", "S:\\"):
    # Guard against FREESURFER_HOME being set to just a drive letter — saw
    # that on this Windows server, leads to bogus 'S:\subjects\fsaverage'
    # being probed.
    YEO_ANNOT_SEARCH.append(Path(_fs_home) / "subjects" / "fsaverage" / "label")
_subj_dir = os.environ.get("SUBJECTS_DIR")
if _subj_dir and _subj_dir not in (".", "/", "\\"):
    YEO_ANNOT_SEARCH.append(Path(_subj_dir) / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(BERN_RECON_ROOT / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(OUT_ROOT / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(SHARED_ROOT / "fsaverage" / "label")  # read-only fallback

def _try_load_yeo(n):
    """Returns (lh_labels, rh_labels, lh_names, rh_names) or None if missing.

    Walks YEO_ANNOT_SEARCH; first directory containing BOTH lh./rh. annots
    wins. Never reads from #SHARE/To_send_collaborators (policy).
    """
    suffix = f"Yeo2011_{n}Networks_N1000.annot"
    for cand in YEO_ANNOT_SEARCH:
        lh_p = cand / f"lh.{suffix}"
        rh_p = cand / f"rh.{suffix}"
        if lh_p.exists() and rh_p.exists():
            print(f"[Yeo{n}] loading from {cand}")
            lh_lbl, _, lh_names = read_annot(str(lh_p))
            rh_lbl, _, rh_names = read_annot(str(rh_p))
            return lh_lbl, rh_lbl, _decode_annot_names(lh_names), _decode_annot_names(rh_names)
    print(f"[WARN] Yeo {n}-network annot not found. Searched:")
    for cand in YEO_ANNOT_SEARCH:
        print(f"         {cand}")
    print(f"       FreeSurfer ships these by default — set FREESURFER_HOME or")
    print(f"       drop them into {OUT_ROOT / 'fsaverage' / 'label'} .")
    print(f"       yeo{n}_network column will be 'Unavailable' until then.")
    return None

YEO7  = _try_load_yeo(7)
YEO17 = _try_load_yeo(17)

def _yeo_at(xyz, hemi_norm, annots):
    """Look up Yeo network label at nearest fsaverage pial vertex."""
    if annots is None:
        return "Unavailable"
    lh_lbl, rh_lbl, lh_names, rh_names = annots
    if hemi_norm == "lh":
        _, v = fs_lh_pialkd.query(xyz)
        lid = int(lh_lbl[v])
        return lh_names[lid] if 0 <= lid < len(lh_names) else "unknown"
    else:
        _, v = fs_rh_pialkd.query(xyz)
        lid = int(rh_lbl[v])
        return rh_names[lid] if 0 <= lid < len(rh_names) else "unknown"

def _dist_to_pial_mm(xyz, hemi_norm):
    """Euclidean distance from xyz to the nearest fsaverage pial vertex."""
    if hemi_norm == "lh":
        d, _ = fs_lh_pialkd.query(xyz)
    else:
        d, _ = fs_rh_pialkd.query(xyz)
    return float(d)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def _cohort(pid: str) -> str:
    if pid.startswith("PAT_"): return "HUG"
    if pid.startswith("EL"):    return "BERN"
    if pid.startswith("MicroEPI") or pid.startswith("G-") or pid.startswith("B-"):
        return "MICROEPI"
    return "UNKNOWN"

def _hemi_from_x(x: float) -> str:
    return "L" if x < 0 else "R"

def _hemi_norm(h: str) -> str:
    h = str(h).upper()
    return "lh" if h.startswith("L") else "rh"

def _col(df, idx, col, fallback_col=None):
    if col in df.columns:
        return df.iloc[idx][col]
    if fallback_col and fallback_col in df.columns:
        return df.iloc[idx][fallback_col]
    return ""

def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    return np.concatenate([im[:H] for im in imgs], axis=1)

def render_fsaverage(points):
    lh = make_mesh(fs_lh_v, fs_lh_f)
    rh = make_mesh(fs_rh_v, fs_rh_f)
    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_mesh(lh, color="#ead6db", opacity=0.25)
        pl.add_mesh(rh, color="#ead6db", opacity=0.25)
        pl.add_points(points, color="purple", point_size=10,
                      render_points_as_spheres=True)
        if   view == "left":    pl.view_yz(negative=True)
        elif view == "right":   pl.view_yz(negative=False)
        else:                    pl.view_xz()
        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)
    return stitch_horiz(views)

# ------------------------------------------------------------
# PROJECTION HELPERS
# ------------------------------------------------------------
def _pat_surface_proj(pid, pts_surface):
    """LEPTOVOX → subject pial nearest-vertex + spherical reg → fsaverage.

    Works for both PAT (subj_dir under SHARED_ROOT) and EL (subj_dir under
    BERN_RECON_ROOT). Cohort routing lives in subj_dir_for(pid) at the top
    of this cell.
    """
    subj_dir = subj_dir_for(pid)
    lh_v, _   = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, _   = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_sph, _ = read_geometry(str(subj_dir / "surf" / "lh.sphere.reg"))
    rh_sph, _ = read_geometry(str(subj_dir / "surf" / "rh.sphere.reg"))
    lh_kd = cKDTree(lh_v); rh_kd = cKDTree(rh_v)

    n = len(pts_surface)
    fs_xyz = np.zeros((n, 3), dtype=float)
    hemi   = ["R"] * n
    for i, p in enumerate(pts_surface):
        dl, il = lh_kd.query(p); dr, ir = rh_kd.query(p)
        if dl <= dr:
            hemi[i] = "L"
            _, fv = fs_lh_sphkd.query(lh_sph[il])
            fs_xyz[i] = fs_lh_v[fv]
        else:
            hemi[i] = "R"
            _, fv = fs_rh_sphkd.query(rh_sph[ir])
            fs_xyz[i] = fs_rh_v[fv]
    return fs_xyz, hemi

def project_patient(pid, df_native):
    """
    Compute fsaverage XYZ + hemi for every contact. Same path for every
    cohort now that both PAT and EL go through LEPTOVOX → tkrRAS.

    Branching is_wm:
      - is_wm == 0 (cortical / grid): spherical registration via subject pial
      - is_wm == 1 (depth white-matter): talairach affine via rs (no surface snap)
    """
    pts = df_native[["x", "y", "z"]].to_numpy(float)
    n   = len(pts)
    is_wm = (df_native["is_wm"].to_numpy(int)
             if "is_wm" in df_native.columns
             else np.zeros(n, dtype=int))

    fs_xyz = np.zeros((n, 3), dtype=float)
    hemi   = ["R"] * n

    depth_idx = np.where(is_wm == 1)[0]
    if len(depth_idx) > 0:
        pts_depth_tal, _ = rs.subject_tkr_to_fsaverage_tkr(pid, pts[depth_idx])
        for j, i in enumerate(depth_idx):
            fs_xyz[i] = pts_depth_tal[j]
            hemi[i]   = _hemi_from_x(float(pts_depth_tal[j, 0]))

    surf_idx = np.where(is_wm == 0)[0]
    if len(surf_idx) > 0:
        surf_xyz, surf_hemi = _pat_surface_proj(pid, pts[surf_idx])
        for j, i in enumerate(surf_idx):
            fs_xyz[i] = surf_xyz[j]
            hemi[i]   = surf_hemi[j]

    return fs_xyz, hemi, is_wm

def build_patient_df(pid, df_native, cohort_str):
    """Wrap projection + Yeo lookup into a unified per-patient DataFrame."""
    fs_xyz, hemi, is_wm = project_patient(pid, df_native)
    rows = []
    for i in range(len(df_native)):
        name = _col(df_native, i, "name", "electrode")
        name_raw = _col(df_native, i, "name_raw") or name
        hn = _hemi_norm(hemi[i])
        is_wm_i = int(is_wm[i])
        is_cortical = 0 if is_wm_i == 1 else 1
        if is_wm_i == 1:
            yeo7  = "WhiteMatter"
            yeo17 = "WhiteMatter"
            dist_pial = float("nan")
        else:
            yeo7  = _yeo_at(fs_xyz[i], hn, YEO7)
            yeo17 = _yeo_at(fs_xyz[i], hn, YEO17)
            dist_pial = _dist_to_pial_mm(fs_xyz[i], hn)
        rows.append({
            "patient":         pid,
            "cohort":          cohort_str,
            "name":            str(name).strip(),
            "name_raw":        str(name_raw).strip(),
            "hemi":            hemi[i],
            "x":               float(fs_xyz[i, 0]),
            "y":               float(fs_xyz[i, 1]),
            "z":               float(fs_xyz[i, 2]),
            "is_wm":           is_wm_i,
            "is_cortical":     is_cortical,
            "dist_to_pial_mm": dist_pial,
            "yeo7_network":    yeo7,
            "yeo17_network":   yeo17,
        })
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# BUILD PATIENT LIST
# ------------------------------------------------------------
PAT_PATIENTS = sorted([p.name for p in OUT_ROOT.iterdir()
                       if p.is_dir() and p.name.startswith("PAT_")])

EL_PATIENTS = [
    "EL030","EL033","EL034","EL035","EL036","EL037","EL038",
    "EL039","EL040","EL042","EL043","EL044","EL045",
]

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []
qc = []

# Unified main loop — every cohort reads its tkrRAS CSV produced by cell 2.
# No more Lookup.xlsx fallback for ELs; the LEPTOVOX path applies to all.
for pid in PAT_PATIENTS + EL_PATIENTS:
    cohort_str = _cohort(pid)
    try:
        csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
        if not csv_in.is_file():
            print(f"[{pid}] SKIP: missing {csv_in} — run cell 2 first.")
            qc.append({"patient": pid, "status": "missing-tkrRAS-csv"})
            continue
        df_native = pd.read_csv(csv_in)

        df_fs = build_patient_df(pid, df_native, cohort_str)

        out_dir = OUT_ROOT / "fsaverage" / "coords"
        out_dir.mkdir(parents=True, exist_ok=True)
        df_fs.to_csv(out_dir / f"{pid}_contacts_fsaverage.csv", index=False)
        all_rows.append(df_fs)

        n_total = len(df_fs)
        n_wm    = int((df_fs["is_wm"] == 1).sum())
        n_cort  = int((df_fs["is_cortical"] == 1).sum())
        qc.append({
            "patient": pid, "status": "OK",
            "n_contacts": n_total, "n_cortical": n_cort, "n_wm": n_wm,
        })
        print(f"[{pid}] {n_total:>4} contacts  cortical={n_cort:>3}  wm={n_wm:>3}  ->  {pid}_contacts_fsaverage.csv")
    except Exception as e:
        qc.append({"patient": pid, "status": f"ERROR: {e}"})
        print(f"[{pid}] ERROR: {e}")

# ------------------------------------------------------------
# AGGREGATES — both variants
# ------------------------------------------------------------
if not all_rows:
    raise RuntimeError("No patients projected successfully — aborting aggregate write.")

df_all = pd.concat(all_rows, ignore_index=True)

out_dir = OUT_ROOT / "fsaverage" / "coords"
out_dir.mkdir(parents=True, exist_ok=True)

out_with_wm = out_dir / "ALL_PATIENTS_contacts_fsaverage.csv"
df_all.to_csv(out_with_wm, index=False)
print(f"\nSaved with-WM aggregate ({len(df_all):>4} rows) -> {out_with_wm.name}")

df_nowm = df_all[df_all["is_wm"] == 0].copy()
out_nowm = out_dir / "ALL_PATIENTS_contacts_fsaverage_nowm.csv"
df_nowm.to_csv(out_nowm, index=False)
print(f"Saved no-WM   aggregate ({len(df_nowm):>4} rows) -> {out_nowm.name}  (252 KEEP_WM=False reads this)")

# QC TSV
pd.DataFrame(qc).to_csv(out_dir / "_regen_qc.tsv", sep="\t", index=False)
print(f"QC table: {(out_dir / '_regen_qc.tsv')}")

# Yeo coverage summary
print("\nYeo 7  distribution:")
print(df_all["yeo7_network"].value_counts())
print("\nYeo 17 distribution (top 20):")
print(df_all["yeo17_network"].value_counts().head(20))

# ------------------------------------------------------------
# GROUP MOSAIC PNG (unchanged behaviour)
# ------------------------------------------------------------
pts_all = df_all[["x", "y", "z"]].to_numpy(float)
mosaic = render_fsaverage(pts_all)
png_dir = OUT_ROOT / "fsaverage" / "png"
png_dir.mkdir(parents=True, exist_ok=True)
png_out = png_dir / "ALL_PATIENTS_fsaverage_mosaic.png"
iio.imwrite(png_out, mosaic)
print(f"Saved group fsaverage mosaic -> {png_out}")


## 250_recon/talairach outputs

In [11]:
# ============================================================
# Volumetric group mapping: patient tkrRAS → Talairach space
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyvista as pv
import imageio.v3 as iio
import re

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
OUT_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon")

PATIENT_COLORS = [
    "blueviolet","fuchsia","deeppink","crimson","pink","red",
    "chocolate","gold","purple","saddlebrown","lemonchiffon",
    "lavenderblush","lime","powderblue","forestgreen","lightcyan",
    "navy","darkslategray","black","darkred","darkolivegreen","aquamarine","aquamarine","aquamarine"
]

def render_volumetric_by_patient(df_all):
    """
    Render Talairach/MNI points with one color per patient
    to visually detect bad transforms.
    """
    pl_views = []

    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        if "patient" not in df_all.columns:
            df_all["patient"] = df_all.get("pid", None)
        df_all["patient"] = df_all["leptovox_file"].str.extract(r"(PAT_\d+)", expand=False)

        for i, (pid, dfp) in enumerate(df_all.groupby("patient")):
            color = PATIENT_COLORS[i % len(PATIENT_COLORS)]
            pts = dfp[["x", "y", "z"]].to_numpy(float)

            pl.add_points(
                pts,
                color=color,
                point_size=10,
                render_points_as_spheres=True,
                opacity=0.9,
                label=pid,
            )

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        pl_views.append(img)

    # stitch horizontally
    H = min(im.shape[0] for im in pl_views)
    return np.concatenate([im[:H] for im in pl_views], axis=1)


# ------------------------------------------------------------
# TAL TRANSFORM PARSER
# ------------------------------------------------------------
def load_talairach_xfm(xfm_path: Path) -> np.ndarray:
    """
    Robust FreeSurfer talairach.xfm / MNI linear transform parser.
    Handles trailing semicolons and format variants.
    Returns 4x4 affine (RAS -> MNI/Talairach).
    """
    lines = xfm_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    start = None
    for i, ln in enumerate(lines):
        if ln.strip().startswith("Linear_Transform"):
            start = i + 1
            break

    if start is None:
        raise RuntimeError(f"Could not find Linear_Transform in {xfm_path}")

    rows = []
    for j in range(3):
        raw = lines[start + j].strip()
        parts = [p.rstrip(";") for p in raw.split()]
        if len(parts) != 4:
            raise RuntimeError(f"Invalid transform row: {raw}")
        rows.append([float(p) for p in parts])

    M = np.eye(4)
    M[:3, :4] = np.array(rows, dtype=float)
    return M


def apply_affine(points_xyz: np.ndarray, M: np.ndarray) -> np.ndarray:
    n = points_xyz.shape[0]
    xyz_h = np.c_[points_xyz, np.ones(n)]
    out = (M @ xyz_h.T).T
    return out[:, :3]


# ------------------------------------------------------------
# RENDERING
# ------------------------------------------------------------
def render_volumetric(points, color="purple"):
    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_points(points, color=color, point_size=10, render_points_as_spheres=True)

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)

    H = min(im.shape[0] for im in views)
    return np.concatenate([im[:H] for im in views], axis=1)


# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []

patient_ids = sorted([
    p.name for p in OUT_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("PAT_")
])

Patients_ignored = ["PAT_5515"]
patient_ids = sorted([pid for pid in patient_ids if pid not in Patients_ignored])

for pid in patient_ids:
    print(f"Talairach mapping: {pid}")

    subj_dir = SHARED_ROOT / pid
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    xfm = subj_dir / "mri" / "transforms" / "talairach.xfm"

    if not csv_in.is_file():
        print(f"  SKIP: missing {csv_in}")
        continue
    if not xfm.is_file():
        print(f"  SKIP: missing {xfm}")
        continue

    # Load data
    df = pd.read_csv(csv_in)
    pts = df[["x", "y", "z"]].to_numpy(float)

    # Load talairach transform
    M = load_talairach_xfm(xfm)

    # Apply transform
    pts_tal = apply_affine(pts, M)

    # Save per-patient CSV
    out_dir = OUT_ROOT / "talairach" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)

    df_out = df.copy()
    df_out[["x", "y", "z"]] = pts_tal
    df_out["space"] = "Talairach"
    df_out["transform"] = "FreeSurfer talairach.xfm"

    out_csv = out_dir / f"{pid}_contacts_talairach.csv"
    df_out.to_csv(out_csv, index=False)

    all_rows.append(df_out)


# # ------------------------------------------------------------
# # CONCATENATE + RENDER GROUP
# # ------------------------------------------------------------
df_all = pd.concat(all_rows, ignore_index=True)

out_all = OUT_ROOT / "talairach" / "coords" / "ALL_PATIENTS_contacts_talairach.csv"
out_all.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(out_all, index=False)

print(f"Saved Talairach group CSV → {out_all}")

# Render combined volumetric view
pts_all = df_all[["x", "y", "z"]].to_numpy(float)
mosaic = render_volumetric(pts_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic.png"
iio.imwrite(png_out, mosaic)

print(f"Saved Talairach mosaic → {png_out}")

mosaic = render_volumetric_by_patient(df_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic_colorcoded.png"
iio.imwrite(png_out, mosaic)

print(f"Saved color-coded Talairach mosaic → {png_out}")



Talairach mapping: PAT_1145
Talairach mapping: PAT_2856
Talairach mapping: PAT_2868
Talairach mapping: PAT_2893
Talairach mapping: PAT_3066
Talairach mapping: PAT_3301
Talairach mapping: PAT_3390
Talairach mapping: PAT_3415
Talairach mapping: PAT_3455
Talairach mapping: PAT_3780
Talairach mapping: PAT_3965
Talairach mapping: PAT_3975
Talairach mapping: PAT_5533
Talairach mapping: PAT_648
Talairach mapping: PAT_6619
Talairach mapping: PAT_6684
Talairach mapping: PAT_6704
Talairach mapping: PAT_6739
Talairach mapping: PAT_6854
Talairach mapping: PAT_699
Talairach mapping: PAT_7045
Saved Talairach group CSV → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon\talairach\coords\ALL_PATIENTS_contacts_talairach.csv
Saved Talairach mosaic → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon\talairach\png\ALL_PATIENTS_talairach_mosaic.png
Saved color-coded Talairach mosaic → \\nasac-m2.unige.ch\m-H

In [12]:
from pathlib import Path
import os

FSAVERAGE_DIR = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage")

def tree(p: Path, max_depth=3, max_items_per_dir=200):
    p = Path(p)
    print(f"\n=== FSAVERAGE TREE ===")
    print(f"root: {p}")
    if not p.exists():
        print("ERROR: path does not exist")
        return

    for root, dirs, files in os.walk(p):
        rel = Path(root).relative_to(p)
        depth = len(rel.parts)
        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        print(f"{indent}{rel if str(rel) != '.' else '.'}/")

        # show dirs
        dirs_sorted = sorted(dirs)
        if dirs_sorted:
            show_dirs = dirs_sorted[:max_items_per_dir]
            for d in show_dirs:
                print(f"{indent}  [D] {d}")
            if len(dirs_sorted) > max_items_per_dir:
                print(f"{indent}  ... ({len(dirs_sorted)-max_items_per_dir} more dirs)")

        # show files
        files_sorted = sorted(files)
        if files_sorted:
            show_files = files_sorted[:max_items_per_dir]
            for f in show_files:
                # show size to spot template volumes etc.
                fp = Path(root) / f
                try:
                    sz = fp.stat().st_size
                except Exception:
                    sz = None
                if sz is None:
                    print(f"{indent}  [F] {f}")
                else:
                    print(f"{indent}  [F] {f}  ({sz/1e6:.1f} MB)")
            if len(files_sorted) > max_items_per_dir:
                print(f"{indent}  ... ({len(files_sorted)-max_items_per_dir} more files)")

tree(FSAVERAGE_DIR, max_depth=4)



=== FSAVERAGE TREE ===
root: \\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage
./
  [D] bem
  [D] label
  [D] mri
  [D] mri.2mm
  [D] scripts
  [D] src
  [D] stats
  [D] surf
  [D] tmp
  [D] touch
  [D] trash
  bem/
  label/
    [F] lh.BA1.label  (0.2 MB)
    [F] lh.BA2.label  (0.3 MB)
    [F] lh.BA3a.label  (0.2 MB)
    [F] lh.BA3b.label  (0.2 MB)
    [F] lh.BA44.label  (0.2 MB)
    [F] lh.BA45.label  (0.1 MB)
    [F] lh.BA4a.label  (0.2 MB)
    [F] lh.BA4p.label  (0.2 MB)
    [F] lh.BA6.label  (0.5 MB)
    [F] lh.MT.label  (0.1 MB)
    [F] lh.Medial_wall.label  (0.6 MB)
    [F] lh.PALS_B12.labels.gii  (0.3 MB)
    [F] lh.PALS_B12_Brodmann.annot  (1.3 MB)
    [F] lh.PALS_B12_Lobes.annot  (1.3 MB)
    [F] lh.PALS_B12_OrbitoFrontal.annot  (1.3 MB)
    [F] lh.PALS_B12_Visuotopic.annot  (1.3 MB)
    [F] lh.V1.label  (0.2 MB)
    [F] lh.V2.label  (0.3 MB)
    [F] lh.Yeo2011_17NetworksConfidence_N1000.mgz  (0.5 MB)
    [F] lh.Yeo2011_17Networks_N1000.annot  (1.3 M

## (dont run) EXTRA: Script to loop through different permutations and also flips to see which fits best (i.e. if a flip or permutation is needed)

In [21]:


# ============================================================
# ONE CELL: LEPTOVOX -> tkrRAS with (perm + flips) search
#          hemi-aware scoring + save TOP-K candidate mosaics
#          so you can visually verify which is correct.
#
# Works with:
#   elec_recon/PAT_XXXX.LEPTOVOX
#   elec_recon/PAT_XXXX.electrodeNames  (first 2 lines are headers)
#   mri/brainmask.mgz (or T1/orig)
#   surf/lh.pial, surf/rh.pial
#
# Outputs per patient under OUT_ROOT/PAT_XXXX/glassbrain/:
#   coords/PAT_XXXX_contacts_tkrRAS.csv
#   png/PAT_XXXX_mosaic_LEPTOVOX.png                (best)
#   png_candidates/PAT_XXXX_cand01_...png ...       (top K)
#   png_candidates/PAT_XXXX_candidates_summary.csv  (table)
# ============================================================

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import nibabel as nib
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree


# -------------------------
# CONFIG
# -------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
OUT_ROOT    = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\250_recon")

# scoring weights
LAMBDA_HEMI = 120.0   # mm penalty for hemisphere mismatch (increase if still mirrored)
K_CANDIDATES_TO_SAVE = 12  # how many candidate mosaics to save per patient

# render params
VIEWS = ("left", "frontal", "right")
WINDOW_SIZE = (1200, 1000)
TRANSPARENT_BG = True
BRAIN_COLOR = "#ead6db"
BRAIN_OPACITY = 0.25
POINT_COLOR = "purple"
POINT_SIZE = 10
POINT_OPACITY = 0.9

OVERWRITE_CSV = True
OVERWRITE_PNG = True
OVERWRITE_CANDIDATES = True


# -------------------------
# IO helpers
# -------------------------
def read_electrode_lines_drop2(path: Path) -> list[str]:
    """Return electrode lines after dropping 2 header lines."""
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 3:
        raise ValueError(f"electrodeNames too short: {path}")
    return lines[2:]

def parse_name_and_hemi(lines: list[str]):
    """
    lines look like: 'FPG1 D L' or 'FOG12 D R'
    We keep the whole line as 'name_raw' for traceability and also extract a clean name + hem label.
    """
    names_raw = []
    names_clean = []
    hemi = []
    for ln in lines:
        parts = ln.split()
        if len(parts) >= 1:
            nm = parts[0]
        else:
            nm = ln
        h = None
        # hem is often last token "L" or "R"
        if len(parts) >= 1:
            last = parts[-1].upper()
            if last in ("L", "R"):
                h = last
        names_raw.append(ln)
        names_clean.append(nm)
        hemi.append(h)
    return np.array(names_raw, dtype=object), np.array(names_clean, dtype=object), np.array(hemi, dtype=object)

def read_leptovox_xyz(path: Path) -> np.ndarray:
    rows = []
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        t = ln.strip()
        if not t or t.startswith("#"):
            continue
        parts = t.replace(",", " ").split()
        if len(parts) < 3:
            continue
        try:
            rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
        except ValueError:
            continue
    if not rows:
        raise ValueError(f"No numeric rows found in {path}")
    return np.asarray(rows, dtype=float)

def pick_mgz(subj_dir: Path) -> Path:
    for cand in ["brainmask.mgz", "T1.mgz", "orig.mgz"]:
        p = subj_dir / "mri" / cand
        if p.is_file():
            return p
    raise FileNotFoundError(f"Missing brainmask/T1/orig in {subj_dir/'mri'}")


# -------------------------
# Geometry / scoring helpers
# -------------------------
def voxel_to_tkr(points_ijk: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    n = points_ijk.shape[0]
    ijk_h = np.c_[points_ijk, np.ones(n)]
    tkr_h = (vox2ras_tkr @ ijk_h.T).T
    return tkr_h[:, :3]

def nearest_pial_metrics(points_tkr: np.ndarray, lh_v: np.ndarray, rh_v: np.ndarray):
    kdl = cKDTree(lh_v)
    kdr = cKDTree(rh_v)
    dl, _ = kdl.query(points_tkr, k=1, workers=-1)
    dr, _ = kdr.query(points_tkr, k=1, workers=-1)
    is_left = dl <= dr
    dist = np.minimum(dl, dr)
    return dist, is_left

def apply_voxel_flips(ijk: np.ndarray, vol_shape, flips: tuple[bool,bool,bool]) -> np.ndarray:
    dims = np.array(vol_shape, dtype=float)
    out = ijk.copy()
    for ax, do_flip in enumerate(flips):
        if do_flip:
            out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
    return out

def hemi_accuracy(pred_is_left: np.ndarray, hemi_expected: np.ndarray) -> float:
    """
    hemi_expected entries are 'L','R', or None
    pred_is_left is boolean array from nearest pial
    """
    mask = np.array([h in ("L","R") for h in hemi_expected], dtype=bool)
    if mask.sum() == 0:
        return np.nan
    exp_is_left = np.array([h == "L" for h in hemi_expected[mask]], dtype=bool)
    pred = pred_is_left[mask]
    return float(np.mean(pred == exp_is_left))

def choose_best_perm_and_flips(
    pts_ijk_raw: np.ndarray,
    vox2ras_tkr: np.ndarray,
    lh_v: np.ndarray,
    rh_v: np.ndarray,
    vol_shape,
    hemi_expected: np.ndarray,
    lambda_hemi: float,
):
    """
    Search over:
      - 6 perms of columns
      - 8 voxel flips (i, j, k)
    Score = median_dist_to_pial + lambda_hemi*(1 - hemi_acc)
      If hemi_expected missing => score = median_dist only.

    Returns sorted list of candidate dicts (best first).
    """
    dims = np.array(vol_shape, dtype=float)

    candidates = []
    perms = list(itertools.permutations([0,1,2], 3))
    flips_list = list(itertools.product([False, True], repeat=3))

    for perm in perms:
        base = pts_ijk_raw[:, perm]

        # early reject if totally out-of-bounds (with slack)
        if not ((base >= -5).all() and (base <= (dims + 5)).all()):
            continue

        for flips in flips_list:
            ijk = apply_voxel_flips(base, vol_shape, flips)

            # bound check again (allow slack)
            if not ((ijk >= -5).all() and (ijk <= (dims + 5)).all()):
                continue

            tkr = voxel_to_tkr(ijk, vox2ras_tkr)
            dist_mm, pred_is_left = nearest_pial_metrics(tkr, lh_v, rh_v)

            med = float(np.median(dist_mm))
            acc = hemi_accuracy(pred_is_left, hemi_expected)

            if np.isnan(acc):
                score = med
            else:
                score = med + lambda_hemi * (1.0 - acc)

            candidates.append({
                "perm": perm,
                "flips": flips,
                "median_dist_mm": med,
                "hemi_acc": acc,
                "score": float(score),
                "tkr": tkr,
                "dist": dist_mm,
                "pred_is_left": pred_is_left,
            })

    if not candidates:
        raise RuntimeError("No valid candidates found (all out-of-bounds).")

    candidates.sort(key=lambda d: d["score"])
    return candidates


# -------------------------
# Rendering helpers
# -------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def set_view(pl, view):
    view = view.lower()
    if view == "left":
        pl.view_yz(negative=True)
    elif view == "frontal":
        pl.view_xz(negative=False)
    elif view == "right":
        pl.view_yz(negative=False)
    else:
        raise ValueError(view)
    pl.camera.zoom(1.15)

def render_view(lh_mesh, rh_mesh, points_tkr, view):
    pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    pl.set_background("white")
    pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
                  point_size=POINT_SIZE, opacity=POINT_OPACITY)
    set_view(pl, view)
    img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
    pl.close()
    return img

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    imgs = [im[:H] for im in imgs]
    return np.concatenate(imgs, axis=1)


# -------------------------
# Core per-patient routine
# -------------------------
def export_and_mosaic_patient(pid: str):
    pid = str(pid)
    subj_dir = SHARED_ROOT / pid

    names_path = subj_dir / "elec_recon" / f"{pid}.electrodeNames"
    vox_path   = subj_dir / "elec_recon" / f"{pid}.LEPTOVOX"
    if not names_path.is_file():
        raise FileNotFoundError(f"missing {names_path}")
    if not vox_path.is_file():
        raise FileNotFoundError(f"missing {vox_path}")

    mgz = pick_mgz(subj_dir)
    img = nib.load(str(mgz))
    vox2ras_tkr = img.header.get_vox2ras_tkr()
    vol_shape = img.shape[:3]

    # surfaces
    lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_mesh = make_mesh(lh_v, lh_f)
    rh_mesh = make_mesh(rh_v, rh_f)

    # names + hemi labels
    lines = read_electrode_lines_drop2(names_path)
    names_raw, names_clean, hemi = parse_name_and_hemi(lines)

    # LEPTOVOX coords
    pts = read_leptovox_xyz(vox_path)
    if pts.shape[0] != len(names_raw):
        raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(names_raw)})")

    # 0/1-based detection
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    dims = np.array(vol_shape, dtype=float)
    looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
    looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()

    pts_ijk_raw = pts.copy()
    index_mode = "0-based (assumed)"
    if looks_one_based and not looks_zero_based:
        pts_ijk_raw -= 1.0
        index_mode = "1-based->0-based"

    # candidate search (perm + flips) with hemi-aware scoring
    cands = choose_best_perm_and_flips(
        pts_ijk_raw, vox2ras_tkr, lh_v, rh_v, vol_shape, hemi,
        lambda_hemi=LAMBDA_HEMI,
    )

    best = cands[0]
    pts_tkr = best["tkr"]
    dist_mm = best["dist"]
    pred_is_left = best["pred_is_left"]

    # outputs
    out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
    out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
    out_candir = OUT_ROOT / pid / "glassbrain" / "png_candidates"
    out_coords.mkdir(parents=True, exist_ok=True)
    out_pngdir.mkdir(parents=True, exist_ok=True)
    out_candir.mkdir(parents=True, exist_ok=True)

    out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
    out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

    # Save best CSV
    if OVERWRITE_CSV or (not out_csv.is_file()):
        df_out = pd.DataFrame({
            "name_raw": names_raw,     # e.g. "FPG1 D L"
            "name": names_clean,       # e.g. "FPG1"
            "hemi_expected": hemi,     # L/R/None
            "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
            "pred_isLeft": pred_is_left.astype(int),
            "dist_to_pial_mm": np.round(dist_mm, 2),
            "source_space": "tkrRAS",
            "source_provenance": f"LEPTOVOX; {index_mode}; perm={best['perm']}; flips={best['flips']}; lambda_hemi={LAMBDA_HEMI}",
            "leptovox_file": str(vox_path),
            "mgz_used": str(mgz),
        })
        df_out.to_csv(out_csv, index=False)

    # Save best mosaic
    if OVERWRITE_PNG or (not out_png.is_file()):
        imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
        mosaic = stitch_horiz(imgs)
        iio.imwrite(out_png, mosaic)

    # Save TOP-K candidate mosaics for visual inspection
    summary_rows = []
    for rank, cand in enumerate(cands[:K_CANDIDATES_TO_SAVE], start=1):
        perm = cand["perm"]
        flips = cand["flips"]
        med = cand["median_dist_mm"]
        acc = cand["hemi_acc"]
        score = cand["score"]

        tag = f"cand{rank:02d}_perm{perm}_flip{tuple(int(b) for b in flips)}_med{med:.2f}_acc{(acc if not np.isnan(acc) else -1):.2f}_score{score:.2f}"
        png_path = out_candir / f"{pid}_{tag}.png"

        if OVERWRITE_CANDIDATES or (not png_path.is_file()):
            imgs = [render_view(lh_mesh, rh_mesh, cand["tkr"], v) for v in VIEWS]
            mosaic = stitch_horiz(imgs)
            iio.imwrite(png_path, mosaic)

        summary_rows.append({
            "rank": rank,
            "perm": perm,
            "flips": flips,
            "median_dist_mm": med,
            "hemi_acc": acc,
            "score": score,
            "png": str(png_path),
        })

    df_sum = pd.DataFrame(summary_rows)
    df_sum.to_csv(out_candir / f"{pid}_candidates_summary.csv", index=False)

    return {
        "pid": pid,
        "status": "OK",
        "n_contacts": int(len(names_raw)),
        "index_mode": index_mode,
        "best_perm": best["perm"],
        "best_flips": best["flips"],
        "median_dist_mm": float(np.median(dist_mm)),
        "hemi_acc": best["hemi_acc"],
        "csv": str(out_csv),
        "png": str(out_png),
        "candidates_dir": str(out_candir),
    }


# -------------------------
# Batch run all PAT_*
# -------------------------
patient_ids = sorted([p.name for p in SHARED_ROOT.iterdir() if p.is_dir() and p.name.startswith("PAT_")])

rows = []
for pid in patient_ids:
    try:
        rows.append(export_and_mosaic_patient(pid))
    except Exception as e:
        rows.append({"pid": pid, "status": "ERROR", "error": str(e)})

df_status = pd.DataFrame(rows)
df_status





,pid,status,n_contacts,index_mode,best_perm,best_flips,median_dist_mm,hemi_acc,csv,png,candidates_dir
0,PAT_1145,OK,142,0-based (assumed),"(0, 1, 2)","(False, False, False)",2.463942,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
1,PAT_2856,OK,176,0-based (assumed),"(0, 1, 2)","(False, False, True)",3.078871,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
2,PAT_2868,OK,63,0-based (assumed),"(2, 0, 1)","(False, True, False)",1.876252,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
3,PAT_2893,OK,173,0-based (assumed),"(0, 2, 1)","(False, True, True)",2.495380,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
4,PAT_3066,OK,200,0-based (assumed),"(0, 1, 2)","(False, False, True)",3.530114,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
5,PAT_3301,OK,202,0-based (assumed),"(0, 1, 2)","(False, False, True)",4.040088,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
6,PAT_3390,OK,118,0-based (assumed),"(1, 0, 2)","(True, True, False)",3.733074,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
7,PAT_3415,OK,148,0-based (assumed),"(0, 1, 2)","(False, False, True)",1.312528,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
8,PAT_3455,OK,116,0-based (assumed),"(0, 2, 1)","(False, True, True)",2.554979,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
9,PAT_3780,OK,136,0-based (assumed),"(1, 0, 2)","(True, True, False)",2.659007,1.000000,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...


In [20]:
df_status.png[0]


'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_Lora\\02_FBM_Clustering\\outputs\\250_recon\\PAT_1145\\glassbrain\\png\\PAT_1145_mosaic_LEPTOVOX.png'